# Sheet 4.3: Serving and deployment

**Roadmap topic 4.3** (Fanout 11.08-11.17, 11.23-11.24). Read first: the FastAPI tutorial (path operations, request body, testing), Docker *Get started*, Kubernetes *Basics*, MLOps Zoomcamp module 4.

Exercises 1-4 have auto-checks (CPU, `pip install fastapi httpx pyyaml`). Exercise 5 deploys on your laptop with Docker and `kind`; no cloud account needed.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells.

In [ ]:
import asyncio, time, math, json, concurrent.futures, numpy as np, yaml
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

def run_async(coro):
    # Runs a coroutine both in plain Python and inside Jupyter/Colab (which already has an event loop)
    with concurrent.futures.ThreadPoolExecutor(1) as ex: return ex.submit(asyncio.run, coro).result()

class FakeLM:
    # Stand-in for your model: deterministic, ~5 ms per batch plus 0.1 ms per item
    def __init__(self): self.calls = 0; self.batch_sizes = []
    def generate_batch(self, prompts, max_tokens):
        self.calls += 1; self.batch_sizes.append(len(prompts)); time.sleep(0.005 + 0.0001 * len(prompts))
        return [p[::-1][:max_tokens] for p in prompts]

## Exercise 1: A validated inference API

Build `make_app(model)` returning a FastAPI app with:
- `GET /health` → `{"status": "ok"}`
- `POST /generate` with body `{"prompt": str, "max_tokens": int = 64}`: `prompt` must be 1-2000 characters, `max_tokens` between 1 and 512 (use pydantic `Field` constraints so FastAPI returns **422** automatically). Response: `{"text": str, "latency_ms": float}`.
- If the prompt contains the string `"<|endoftext|>"`, return **400** with detail `"forbidden token"` (a real-world input-sanitising rule).
- Count requests in `app.state.n_requests`.

In [ ]:
def make_app(model):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    c = TestClient(make_app(FakeLM()))
    assert c.get("/health").json() == {"status": "ok"}
    r = c.post("/generate", json={"prompt": "hello", "max_tokens": 3}); assert r.status_code == 200 and r.json()["text"] == "oll"
    assert isinstance(r.json()["latency_ms"], float)
    assert c.post("/generate", json={"prompt": "hi"}).json()["text"] == "ih", "default max_tokens"
    for bad in ({"prompt": ""}, {"prompt": "x" * 2001}, {"prompt": "a", "max_tokens": 0}, {"prompt": "a", "max_tokens": 513}, {"max_tokens": 5}):
        assert c.post("/generate", json=bad).status_code == 422, f"should reject {str(bad)[:40]}"
    r = c.post("/generate", json={"prompt": "a <|endoftext|> b"}); assert r.status_code == 400 and r.json()["detail"] == "forbidden token"
    assert c.app.state.n_requests == 3
    print("PASS")
_check1()

## Exercise 2: Dynamic batching

GPUs are far more efficient on batches, but requests arrive one at a time. A micro-batcher collects concurrent requests for up to `max_wait_ms` or until `max_batch` items, runs **one** model call, and returns each caller its own result. vLLM and Triton do a (much fancier) version of this.

Implement `Batcher(model, max_batch, max_wait_ms)` with `async submit(prompt, max_tokens)`. Run the model call in a thread (`loop.run_in_executor`) so the event loop isn't blocked. Requests in one batch may have different `max_tokens`: call the model with the batch's maximum and truncate each result.

In [ ]:
class Batcher:
    def __init__(self, model, max_batch=16, max_wait_ms=10):
        # YOUR CODE
        raise NotImplementedError
    async def submit(self, prompt, max_tokens=64):
        # YOUR CODE
        raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    async def scenario():
        lm = FakeLM(); b = Batcher(lm, max_batch=16, max_wait_ms=20)
        prompts = [f"req-{i:03d}" for i in range(64)]; mts = [3 + i % 5 for i in range(64)]
        t = time.perf_counter(); outs = await asyncio.gather(*(b.submit(p, m) for p, m in zip(prompts, mts)))
        burst = time.perf_counter() - t
        t = time.perf_counter(); single = await b.submit("alone", 10); lone = time.perf_counter() - t
        return lm, prompts, mts, outs, burst, single, lone
    lm, prompts, mts, outs, burst, single, lone = run_async(scenario())
    assert outs == [p[::-1][:m] for p, m in zip(prompts, mts)], "results must go back to the right caller, truncated to their max_tokens"
    assert lm.calls <= 6 and max(lm.batch_sizes) <= 16, f"{lm.calls} model calls with sizes {lm.batch_sizes}"
    assert single == "enola" and lone < 0.2, f"a lone request must be served after max_wait, took {lone*1000:.0f} ms"
    print(f"PASS  64 requests served in {lm.calls - 1} model calls ({burst*1000:.0f} ms); lone request {lone*1000:.0f} ms")
_check2()

**Written (2a):** What does `max_wait_ms` trade off? For an LLM, why is request-level batching like this still much worse than vLLM's *continuous batching*?

**Your answer:**

## Exercise 3: Read a load test

`summarise(latencies_s, start_times_s)` returns `dict(p50, p95, p99, throughput_rps, mean_concurrency)` where percentiles are in **milliseconds** (use `np.percentile`, linear interpolation), throughput is completed requests divided by the span from the first start to the last finish, and mean concurrency follows **Little's law** (throughput × mean latency).

In [ ]:
def summarise(latencies_s, start_times_s):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    r = np.random.default_rng(0); n = 2000
    start = np.sort(r.uniform(0, 100, n)); lat = r.lognormal(np.log(0.2), 0.5, n)
    s = summarise(lat, start)
    assert abs(s["p50"] - np.percentile(lat, 50) * 1000) < 1e-6 and s["p99"] > s["p95"] > s["p50"]
    assert abs(s["throughput_rps"] - n / ((start + lat).max() - start.min())) < 1e-9
    assert abs(s["mean_concurrency"] - s["throughput_rps"] * lat.mean()) < 1e-9
    print(f"PASS  p50 {s['p50']:.0f} ms, p95 {s['p95']:.0f} ms, p99 {s['p99']:.0f} ms, {s['throughput_rps']:.1f} req/s, ~{s['mean_concurrency']:.1f} in flight")
_check3()

## Exercise 4: Container and Kubernetes manifests

Write three strings:
- `DOCKERFILE`: **multi-stage** (a builder stage that installs dependencies into a venv or wheels, and a slim runtime stage that copies them), runs as a **non-root** `USER`, `EXPOSE 8000`, a `HEALTHCHECK` hitting `/health`, and starts `uvicorn app:app --host 0.0.0.0 --port 8000`.
- `K8S`: a multi-document YAML with a `Deployment` (2 replicas, container port 8000, CPU/memory **requests and limits**, `readinessProbe` and `livenessProbe` on `/health`), a `Service` (port 80 → 8000), and an `autoscaling/v2` `HorizontalPodAutoscaler` (min 2, max 6, target CPU 70%).

In [ ]:
DOCKERFILE = '''
# YOUR DOCKERFILE
'''
K8S = '''
# YOUR MANIFESTS
'''

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    lines = [l.strip() for l in DOCKERFILE.strip().splitlines() if l.strip() and not l.strip().startswith("#")]
    froms = [l for l in lines if l.upper().startswith("FROM ")]
    assert len(froms) >= 2 and any("COPY --from" in l for l in lines), "must be multi-stage"
    users = [l for l in lines if l.upper().startswith("USER ")]
    assert users and users[-1].split()[1] not in ("root", "0"), "must run as non-root"
    assert any(l.upper().startswith("EXPOSE 8000") for l in lines) and any(l.upper().startswith("HEALTHCHECK") and "/health" in l for l in lines)
    assert "uvicorn" in lines[-1] and "0.0.0.0" in lines[-1]
    docs = {d["kind"]: d for d in yaml.safe_load_all(K8S) if d}
    dep = docs["Deployment"]; c = dep["spec"]["template"]["spec"]["containers"][0]
    assert dep["spec"]["replicas"] == 2 and c["ports"][0]["containerPort"] == 8000
    for k in ("requests", "limits"): assert {"cpu", "memory"} <= set(c["resources"][k]), f"resources.{k}"
    for p in ("readinessProbe", "livenessProbe"): assert c[p]["httpGet"]["path"] == "/health", p
    assert dep["spec"]["selector"]["matchLabels"].items() <= dep["spec"]["template"]["metadata"]["labels"].items(), "selector must match pod labels"
    svc = docs["Service"]; assert svc["spec"]["ports"][0]["port"] == 80 and svc["spec"]["ports"][0]["targetPort"] == 8000
    assert svc["spec"]["selector"].items() <= dep["spec"]["template"]["metadata"]["labels"].items(), "service selector must match pods"
    h = docs["HorizontalPodAutoscaler"]; assert h["apiVersion"] == "autoscaling/v2"
    assert (h["spec"]["minReplicas"], h["spec"]["maxReplicas"]) == (2, 6) and h["spec"]["scaleTargetRef"]["name"] == dep["metadata"]["name"]
    assert h["spec"]["metrics"][0]["resource"]["target"]["averageUtilization"] == 70
    print("PASS")
_check4()

## Exercise 5 (laptop): ship it

1. Put Exercise 1's app (with Exercise 2's batcher) around your Step 3 model: merged + GGUF via `llama-cpp-python`, or a small HF model on CPU. Write `requirements.txt`, build the Exercise 4 image, check it's under 2 GB (`docker images`).
2. `locust` load test at 1, 5 and 20 concurrent users for 2 minutes each. Use Exercise 3 to report p50/p95/p99 and throughput, with and without batching. Identify the bottleneck.
3. `kind create cluster`, load the image (`kind load docker-image`), install metrics-server, `kubectl apply` your manifests. Run the load test against the Service and show the HPA scaling up (`kubectl get hpa -w`) and back down.
4. Continuous delivery: a GitHub Actions workflow that builds and pushes the image to GitHub Container Registry on every push to `main` and tags it with the commit SHA. (Deploying to a real cluster from CI needs a cloud cluster; `kind` stays local, so document the `kubectl set image` step you'd run.)

**Done when** a commit to main produces a new image tag automatically, and your README has the load-test table and an HPA screenshot.

## Checkpoint
Explain, component by component, what happens between a user's HTTP request and a token coming back from your deployment, and where you'd add caching, batching or a queue under 10× load.

---
### Hints for self-grading
- 2a: longer waits give bigger batches (throughput) but add latency to every request, especially at low load. Generation lengths vary, so a request-level batch is held until its longest member finishes; continuous batching adds and removes sequences at every decoding step, and paged KV cache avoids memory fragmentation.